In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

model_input_path = "/content/drive/MyDrive/Marketing_Intelligence_Model_Monitoring/data/outputs/mmm_model_input.csv"

model_data = pd.read_csv(model_input_path, parse_dates=["date"])

print("Dataset loaded successfully!")
print("Shape:", model_data.shape)
print("\nColumns:")
print(model_data.columns.tolist())

model_data.head()

In [ ]:
features = [
    "Google Ads_saturation",
    "Meta Ads_saturation",
    "TikTok Ads_saturation",
    "promotion_active",
    "season_sin",
    "season_cos"
]

missing_columns = [
    col for col in features + ["revenue"]
    if col not in model_data.columns
]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    X = model_data[features].values
    y = model_data["revenue"].values

    print("Features loaded:", X.shape)
    print("Revenue loaded:", y.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

X_scaler = StandardScaler()
y_scaler = StandardScaler()

X_scaled = X_scaler.fit_transform(X)
y_scaled = y_scaler.fit_transform(y.reshape(-1, 1)).flatten()

print("X scaled:", X_scaled.shape)
print("y scaled:", y_scaled.shape)

In [ ]:
!pip install pymc arviz -q

In [ ]:
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

print("PyMC version:", pm.__version__)

In [ ]:
with pm.Model() as bayesian_mmm:

    # Priors
    intercept = pm.Normal("intercept", mu=0, sigma=1)

    beta = pm.Normal(
        "beta",
        mu=0,
        sigma=1,
        shape=X_scaled.shape[1]
    )

    sigma = pm.HalfNormal("sigma", sigma=1)

    # Expected revenue
    mu = intercept + pm.math.dot(X_scaled, beta)

    # Likelihood
    revenue_obs = pm.Normal(
        "revenue_obs",
        mu=mu,
        sigma=sigma,
        observed=y_scaled
    )

    # Posterior sampling
    trace = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.90,
        random_seed=42,
        return_inferencedata=True
    )

In [ ]:
summary = az.summary(
    trace,
    var_names=["intercept", "beta", "sigma"],
    hdi_prob=0.95
)

summary

In [ ]:
az.plot_posterior(
    trace,
    var_names=["beta"],
    hdi_prob=0.95
)

plt.tight_layout()
plt.show()

In [ ]:
diagnostics = az.summary(
    trace,
    var_names=["intercept", "beta", "sigma"]
)

print("Maximum R-hat:", diagnostics["r_hat"].max())
print("Minimum effective sample size:", diagnostics["ess_bulk"].min())

print("Divergences:",
      trace.sample_stats["diverging"].sum().values)

In [ ]:
with bayesian_mmm:
    ppc = pm.sample_posterior_predictive(
        trace,
        random_seed=42
    )

In [ ]:
az.plot_ppc(
    ppc,
    num_pp_samples=100
)

plt.tight_layout()
plt.show()

In [ ]:
feature_names = features

beta_summary = az.summary(
    trace,
    var_names=["beta"],
    hdi_prob=0.95
)

beta_summary["feature"] = feature_names

beta_summary

In [ ]:
import os

OUTPUT_PATH = "/content/drive/MyDrive/Marketing_Intelligence_Model_Monitoring/data/outputs"

beta_summary.to_csv(
    f"{OUTPUT_PATH}/bayesian_mmm_coefficients.csv",
    index=False
)

az.to_netcdf(
    trace,
    f"{OUTPUT_PATH}/bayesian_mmm_trace.nc"
)

print("Bayesian model results saved successfully.")